# V-JEPA 2.1 Probe for Tool Wear Detection

This notebook extracts V-JEPA embeddings from spectrogram videos and trains a linear probe to predict tool wear state.

In [7]:
# Cell 1 — Imports & paths
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from transformers import AutoVideoProcessor, AutoModel

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import umap

# Add ToolWear to path so we can import tool_wear_detection
sys.path.insert(0, str(Path(".").resolve()))
from tool_wear_detection import make_spectrogram_image, _wear_to_label

ARCHIVE_DIR = Path("archive")
OUTPUT_DIR  = Path("jepa_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

WEAR_CLASSES  = ["New (<100µm)", "Worn (100–200µm)", "Severe (>200µm)"]
CUTTERS_TRAIN = ["c1", "c4"]
CUTTERS_TEST  = ["c6"]
ALL_CUTTERS   = ["c1", "c4", "c6"]
FS            = 50_000   # PHM 2010 sampling rate
FPS           = 10
FRAME_SIZE    = 224      # V-JEPA 2.1 input resolution

In [8]:
# Cell 2 — Load wear labels for all cutters
def load_wear_labels(cutter: str) -> dict:
    """
    Returns {cut_index (1-based int): wear_label (0/1/2)} for a cutter.
    wear_um is the mean of the 3 flute measurements per cut.
    """
    wear_csv = ARCHIVE_DIR / cutter / f"{cutter}_wear.csv"
    df = pd.read_csv(wear_csv)  # columns: cut, flute_1, flute_2, flute_3
    result = {}
    for _, row in df.iterrows():
        wear_um = (row["flute_1"] + row["flute_2"] + row["flute_3"]) / 3.0
        result[int(row["cut"])] = _wear_to_label(wear_um)
    return result

# Quick sanity check
labels_c1 = load_wear_labels("c1")
print(f"c1: {len(labels_c1)} cuts, classes present: {set(labels_c1.values())}")
# Expected: c1: 315 cuts, classes present: {0, 1, 2}  (exact set may vary by cutter)


c1: 315 cuts, classes present: {0, 1}


In [9]:
# Cell 3 — Build one video frame (224×224 RGB numpy array)
def render_frame(cutter: str, cut_idx: int, wear_label: int,
                 wear_um: float, total_cuts: int = 315) -> np.ndarray:
    """
    Loads the cut CSV, generates spectrogram, upscales to 224×224,
    overlays wear bar + label text. Returns uint8 RGB array (224, 224, 3).
    """
    # ── Load signal ──────────────────────────────────────────────────
    cut_num  = str(cut_idx).zfill(3)
    cutter_n = cutter[1]  # "c1" -> "1"
    csv_path = ARCHIVE_DIR / cutter / cutter / f"c_{cutter_n}_{cut_num}.csv"
    sig      = pd.read_csv(csv_path, header=None).values  # (127399, 7)

    current = sig[:, 0].astype(np.float32)   # force_x as current proxy
    voltage = 0.8 * current + 0.05 * np.random.randn(len(current)).astype(np.float32)

    # ── Generate spectrogram (3, 64, 64) → upscale to (224, 224, 3) ─
    spec = make_spectrogram_image(current, voltage, fs=FS, img_size=64)
    spec_hwc = np.transpose(spec, (1, 2, 0))             # (64, 64, 3)
    spec_uint8 = (spec_hwc * 255).clip(0, 255).astype(np.uint8)
    img = Image.fromarray(spec_uint8).resize((FRAME_SIZE, FRAME_SIZE),
                                             Image.NEAREST)

    # ── Wear progress bar (bottom 10px) ──────────────────────────────
    BAR_H     = 10
    progress  = cut_idx / total_cuts
    bar_colors = [(0, 200, 80), (255, 200, 0), (220, 50, 50)]  # green/yellow/red
    bar_color = bar_colors[wear_label]

    draw = ImageDraw.Draw(img)
    draw.rectangle([0, FRAME_SIZE - BAR_H, FRAME_SIZE, FRAME_SIZE], fill=(40, 40, 40))
    draw.rectangle([0, FRAME_SIZE - BAR_H, int(FRAME_SIZE * progress), FRAME_SIZE],
                   fill=bar_color)

    # ── Text label (top-left) ─────────────────────────────────────────
    label_text = f"{cutter} · cut {cut_idx:03d} · {WEAR_CLASSES[wear_label]} · {wear_um:.0f}µm"
    draw.rectangle([0, 0, FRAME_SIZE, 14], fill=(0, 0, 0, 180))
    draw.text((2, 1), label_text, fill=(255, 255, 255))

    return np.array(img)

# Quick visual check
labels_c1 = load_wear_labels("c1")
wear_csv   = pd.read_csv(ARCHIVE_DIR / "c1" / "c1_wear.csv")
cut50_um   = wear_csv[wear_csv["cut"] == 50][["flute_1","flute_2","flute_3"]].values.mean()
test_frame = render_frame("c1", 50, labels_c1[50], cut50_um)
plt.figure(figsize=(3, 3))
plt.imshow(test_frame)
plt.axis("off")
plt.title("Frame check: c1 cut 050")
plt.show()
print(f"Frame shape: {test_frame.shape}, dtype: {test_frame.dtype}")
# Expected: Frame shape: (224, 224, 3), dtype: uint8


Frame shape: (224, 224, 3), dtype: uint8


In [10]:
# Cell 4 — Generate one MP4 per training cutter
def generate_video(cutter: str) -> Path:
    """
    Writes jepa_outputs/<cutter>_wear_video.mp4.
    Returns the output path.
    """
    out_path = OUTPUT_DIR / f"{cutter}_wear_video.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    writer   = cv2.VideoWriter(str(out_path), fourcc, FPS,
                               (FRAME_SIZE, FRAME_SIZE))

    wear_csv = pd.read_csv(ARCHIVE_DIR / cutter / f"{cutter}_wear.csv")
    wear_labels = load_wear_labels(cutter)

    for cut_idx in range(1, 316):
        row     = wear_csv[wear_csv["cut"] == cut_idx]
        wear_um = row[["flute_1","flute_2","flute_3"]].values.mean()
        label   = wear_labels[cut_idx]
        frame   = render_frame(cutter, cut_idx, label, wear_um)
        bgr     = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        writer.write(bgr)

        if cut_idx % 50 == 0:
            print(f"  {cutter} cut {cut_idx:03d}/315 — {WEAR_CLASSES[label]}")

    writer.release()
    size_mb = out_path.stat().st_size / 1e6
    print(f"Saved {out_path} ({size_mb:.1f} MB)")
    return out_path

for cutter in ALL_CUTTERS:
    print(f"\nGenerating video for {cutter}...")
    generate_video(cutter)

print("\nAll videos done.")



Generating video for c1...
  c1 cut 050/315 — New (<100µm)
  c1 cut 100/315 — New (<100µm)
  c1 cut 150/315 — New (<100µm)
  c1 cut 200/315 — Worn (100–200µm)
  c1 cut 250/315 — Worn (100–200µm)
  c1 cut 300/315 — Worn (100–200µm)
Saved jepa_outputs/c1_wear_video.mp4 (1.1 MB)

Generating video for c4...
  c4 cut 050/315 — New (<100µm)
  c4 cut 100/315 — New (<100µm)
  c4 cut 150/315 — New (<100µm)
  c4 cut 200/315 — New (<100µm)
  c4 cut 250/315 — Worn (100–200µm)
  c4 cut 300/315 — Worn (100–200µm)
Saved jepa_outputs/c4_wear_video.mp4 (1.1 MB)

Generating video for c6...
  c6 cut 050/315 — New (<100µm)
  c6 cut 100/315 — Worn (100–200µm)
  c6 cut 150/315 — Worn (100–200µm)
  c6 cut 200/315 — Worn (100–200µm)
  c6 cut 250/315 — Worn (100–200µm)
  c6 cut 300/315 — Severe (>200µm)
Saved jepa_outputs/c6_wear_video.mp4 (1.2 MB)

All videos done.


In [11]:
# Cell 5 — Load pretrained V-JEPA 2.1 encoder (frozen)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID   = "facebook/vjepa2-vitl-fpc64-256"

print(f"Device: {DEVICE}")
print(f"Loading {MODEL_ID} ...")

processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
vjepa     = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
vjepa.eval()

# Freeze all parameters — pure feature extractor
for p in vjepa.parameters():
    p.requires_grad = False

total_params = sum(p.numel() for p in vjepa.parameters()) / 1e6
print(f"Loaded. Parameters: {total_params:.0f}M (frozen)")
print(f"Embedding dim: 1024 (ViT-L)")

Device: cuda
Loading facebook/vjepa2-vitl-fpc64-256 ...


Loading weights: 100%|██████████| 587/587 [00:00<00:00, 19953.13it/s]

Loaded. Parameters: 326M (frozen)
Embedding dim: 1024 (ViT-L)


In [12]:
# Cell 6 — Extract per-frame embeddings from each video
CHUNK_SIZE = 32   # frames per forward pass (tune down if OOM)

def video_to_frames_tensor(video_path: Path) -> torch.Tensor:
    """
    Reads MP4, returns float32 tensor (T, 3, 224, 224) normalised to [0,1].
    """
    cap    = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        frames.append(t)
    cap.release()
    return torch.stack(frames)  # (T, 3, H, W)

def extract_embeddings(video_path: Path) -> np.ndarray:
    """
    Returns (T, 1024) mean-pooled embeddings for every frame in the video.
    Uses AutoVideoProcessor to prepare inputs correctly for VJEPA2Model.
    """
    frames = video_to_frames_tensor(video_path)  # (T, 3, 224, 224)
    T      = frames.shape[0]
    embeds = []

    for start in range(0, T, CHUNK_SIZE):
        chunk    = frames[start:start + CHUNK_SIZE]  # (<=32, 3, 224, 224)
        # processor expects list of uint8 numpy arrays (H, W, C)
        chunk_np = (chunk.permute(0, 2, 3, 1).numpy() * 255).astype(np.uint8)
        inputs   = processor(list(chunk_np), return_tensors="pt")
        inputs   = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = vjepa(**inputs)
        # last_hidden_state: (1, tokens, 1024) — mean pool per frame
        hidden           = out.last_hidden_state[0]   # (tokens, 1024)
        n_frames_in_chunk = chunk.shape[0]
        tokens_per_frame  = hidden.shape[0] // n_frames_in_chunk
        for f in range(n_frames_in_chunk):
            frame_tokens = hidden[f * tokens_per_frame:(f + 1) * tokens_per_frame]
            embeds.append(frame_tokens.mean(dim=0).cpu().numpy())

    return np.stack(embeds)  # (T, 1024)

# Extract for all cutters
all_embeddings = []
all_labels     = []
all_cutters_id = []

for cutter in ALL_CUTTERS:
    video_path  = OUTPUT_DIR / f"{cutter}_wear_video.mp4"
    wear_labels = load_wear_labels(cutter)

    print(f"Extracting embeddings: {cutter} ...")
    emb = extract_embeddings(video_path)   # (315, 1024)
    lbl = np.array([wear_labels[i] for i in range(1, 316)])

    all_embeddings.append(emb)
    all_labels.append(lbl)
    all_cutters_id.extend([cutter] * 315)
    print(f"  {cutter}: embeddings {emb.shape}, classes {np.unique(lbl)}")

embeddings = np.vstack(all_embeddings)   # (945, 1024)
labels     = np.concatenate(all_labels)  # (945,)
cutter_ids = np.array(all_cutters_id)   # (945,)

print(f"\nFull matrix: {embeddings.shape}, labels: {labels.shape}")
# Expected: Full matrix: (945, 1024), labels: (945,)

Extracting embeddings: c1 ...
  c1: embeddings (315, 1024), classes [0 1]
Extracting embeddings: c4 ...
  c4: embeddings (315, 1024), classes [0 1 2]
Extracting embeddings: c6 ...
  c6: embeddings (315, 1024), classes [0 1 2]

Full matrix: (945, 1024), labels: (945,)


In [13]:
# Cell 7 — Linear probe: train on c1+c4, test on c6
train_mask = (cutter_ids == "c1") | (cutter_ids == "c4")
test_mask  = cutter_ids == "c6"

X_train, y_train = embeddings[train_mask], labels[train_mask]
X_test,  y_test  = embeddings[test_mask],  labels[test_mask]

# Standardise — important for logistic regression on high-dim embeddings
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train, y_train)

y_pred   = clf.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f"Overall accuracy on c6 (held-out): {accuracy:.3f}")

print("\nPer-class report:")
print(classification_report(y_test, y_pred, target_names=WEAR_CLASSES))

# CNN baseline reference
print("CNN baseline: New 0.940  Worn 0.910  Severe 0.920")


Overall accuracy on c6 (held-out): 0.673

Per-class report:
                  precision    recall  f1-score   support

    New (<100µm)       0.45      1.00      0.62        63
Worn (100–200µm)       0.86      0.66      0.74       227
 Severe (>200µm)       0.00      0.00      0.00        25

        accuracy                           0.67       315
       macro avg       0.43      0.55      0.45       315
    weighted avg       0.71      0.67      0.66       315

CNN baseline: New 0.940  Worn 0.910  Severe 0.920


In [14]:
# Cell 8 — Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=WEAR_CLASSES,
            yticklabels=WEAR_CLASSES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("V-JEPA 2.1 Linear Probe — Confusion Matrix (test: c6)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")

Saved confusion_matrix.png


In [15]:
# Cell 9 — UMAP: coloured by wear class and by cutter
print("Fitting UMAP on all 945 embeddings (may take ~60s)...")
reducer  = umap.UMAP(n_components=2, random_state=42, n_neighbors=20, min_dist=0.1)
emb_2d   = reducer.fit_transform(StandardScaler().fit_transform(embeddings))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# — Plot 1: coloured by wear class —
class_colors = ["#00cc55", "#ffcc00", "#ee3333"]
for cls_id, (cls_name, color) in enumerate(zip(WEAR_CLASSES, class_colors)):
    mask = labels == cls_id
    axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                    c=color, label=cls_name, s=8, alpha=0.7)
axes[0].set_title("UMAP — coloured by wear class")
axes[0].legend(markerscale=3, fontsize=8)
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")

# — Plot 2: coloured by cutter —
cutter_color_map = {"c1": "#4488ff", "c4": "#ff8844", "c6": "#88cc44"}
for cutter in ALL_CUTTERS:
    mask = cutter_ids == cutter
    axes[1].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                    c=cutter_color_map[cutter], label=cutter, s=8, alpha=0.7)
axes[1].set_title("UMAP — coloured by cutter")
axes[1].legend(markerscale=3, fontsize=8)
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")

plt.suptitle("V-JEPA 2.1 Embedding Space — PHM 2010 Tool Wear")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap.png", dpi=150)
plt.show()
print("Saved umap.png")

Fitting UMAP on all 945 embeddings (may take ~60s)...
Saved umap.png


In [16]:
/l# Cell 10 — Per-frame embedding drift (L2 distance between consecutive frames)
fig, axes = plt.subplots(len(ALL_CUTTERS), 1, figsize=(12, 8), sharex=False)

for ax, cutter in zip(axes, ALL_CUTTERS):
    mask     = cutter_ids == cutter
    emb_c    = embeddings[mask]          # (315, 1024)
    lbl_c    = labels[mask]              # (315,)

    diffs = np.linalg.norm(np.diff(emb_c, axis=0), axis=1)  # (314,)
    cuts  = np.arange(2, 316)

    # Background colour bands by wear class
    wear_transitions = np.where(np.diff(lbl_c))[0] + 1  # cut indices where class changes
    class_colors_bg  = ["#00cc5522", "#ffcc0022", "#ee333322"]
    prev = 0
    for t in list(wear_transitions) + [315]:
        cls = lbl_c[prev]
        ax.axvspan(prev + 1, t, color=class_colors_bg[cls])
        prev = t

    ax.plot(cuts, diffs, lw=0.8, color="#4488ff", alpha=0.9)
    ax.set_title(f"{cutter} — embedding drift (L2 consecutive frames)")
    ax.set_ylabel("L2 distance")
    ax.set_xlabel("Cut index")

    # Annotate class transitions
    for t in wear_transitions:
        ax.axvline(t + 1, color="white", lw=1.2, linestyle="--", alpha=0.7)
        ax.text(t + 1, diffs.max() * 0.9, WEAR_CLASSES[lbl_c[t]],
                fontsize=6, color="white", rotation=90, va="top")

plt.suptitle("V-JEPA 2.1 — Embedding Drift Across Wear Progression")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "embedding_drift.png", dpi=150)
plt.show()
print("Saved embedding_drift.png")


Saved embedding_drift.png


In [17]:
# Cell 11 — Results summary
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1, 2])

print("=" * 55)
print("V-JEPA 2.1 Linear Probe Results (test cutter: c6)")
print("=" * 55)
print(f"{'Class':<22} {'Precision':>9} {'Recall':>7} {'F1':>6} {'N':>5}")
print("-" * 55)
for i, cls in enumerate(WEAR_CLASSES):
    print(f"{cls:<22} {precision[i]:>9.3f} {recall[i]:>7.3f} {f1[i]:>6.3f} {support[i]:>5}")
print("-" * 55)
print(f"{'Overall accuracy':<22} {accuracy:>9.3f}")
print()
print("CNN baseline:          New 0.940  Worn 0.910  Severe 0.920")
print()
print("Output files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f}")

V-JEPA 2.1 Linear Probe Results (test cutter: c6)
Class                  Precision  Recall     F1     N
-------------------------------------------------------
New (<100µm)               0.447   1.000  0.618    63
Worn (100–200µm)           0.856   0.656  0.743   227
Severe (>200µm)            0.000   0.000  0.000    25
-------------------------------------------------------
Overall accuracy           0.673

CNN baseline:          New 0.940  Worn 0.910  Severe 0.920

Output files:
  jepa_outputs/c1_wear_video.mp4
  jepa_outputs/c4_wear_video.mp4
  jepa_outputs/c6_wear_video.mp4
  jepa_outputs/confusion_matrix.png
  jepa_outputs/embedding_drift.png
  jepa_outputs/umap.png
